<a href="https://colab.research.google.com/github/drmalalaphd/NLP-Lingala-Dataset-from-Pop-The-Balloon-Congo-/blob/main/ep19_first_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lingála test run: Pop the Balloon, Episode 19
Run the cells top to bottom. Before cell 3: Runtime > Change runtime type > T4 GPU.

In [ ]:
!pip install -q yt-dlp faster-whisper

In [ ]:
URL = "https://youtube.com/watch?v=ARak_ENt6ao"
START, END = "0:00", "19:00"   # first 19 minutes, jingle included
OFFSET = 0                     # seconds; set to START in seconds if START is not 0:00
LANG = None                    # None = Whisper guesses; try "ln" (Lingala) or "fr" to compare

!yt-dlp -x --audio-format wav --download-sections "*{START}-{END}" -o "ep19.%(ext)s" "{URL}"

In [ ]:
import csv, os, subprocess
from faster_whisper import WhisperModel

model = WhisperModel("large-v3", device="cuda", compute_type="float16")
segments, info = model.transcribe("ep19.wav", language=LANG, vad_filter=True, beam_size=5)
print("Whisper language guess:", info.language, round(info.language_probability, 2))

os.makedirs("clips", exist_ok=True)
fields = ["clip_id","audio_file","source","start","end","transcript","whisper_language_guess",
          "language","speaker_id","speaker_gender","laughter","background_noise",
          "overlapping_speech","usable","checked_by"]
rows = []
for i, seg in enumerate(segments, 1):
    cid = f"popballoon_ep19_{i:04d}"
    path = f"clips/{cid}.wav"
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i","ep19.wav",
                    "-ss",str(seg.start),"-to",str(seg.end),path], check=True)
    rows.append({"clip_id":cid, "audio_file":path, "source":URL,
                 "start":round(seg.start+OFFSET,1), "end":round(seg.end+OFFSET,1),
                 "transcript":seg.text.strip(), "whisper_language_guess":info.language,
                 "language":"", "speaker_id":"", "speaker_gender":"", "laughter":"",
                 "background_noise":"", "overlapping_speech":"", "usable":"", "checked_by":""})

with open("ep19_clips.csv","w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(rows)
print(len(rows), "clips saved")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/Into Binary Research Institute/Projects/Dataset Projects/ep19_first_test"
!mkdir -p "{dest}"
!cp -r clips ep19_clips.csv "{dest}"
print("Saved to", dest)